# SILVER: check, clean and type the data

Takes the raw Bronze rows and:

1. Labels every row as valid or rejected, with the reason.
2. Saves the valid rows with proper data types (numbers, dates) and a line amount.
3. Saves the rejected rows in their own table, so nothing disappears without a record.

## Step 1: label every row

A row is rejected if it has no customer ID, belongs to a cancelled order (invoice starts with C), or uses a non-product code such as a manual adjustment or bank charge.

In [ ]:
%sql
create or replace temporary view checked_rows as
select
    *,
    case
            when customer_id is null or trim(customer_id) = '' then 'missing customer id'
            when invoice like 'C%' then 'cancelled order'
            when stock_code in ('M', 'D', 'BANK CHARGES', 'ADJUST', 'AMAZONFEE', 'TEST001', 'CRUK') then 'non-product code'
            else 'valid'
        end as row_status
from procat.retail.bronze_order_lines;

## Step 2: save the valid rows with proper types

In [ ]:
%sql
create or replace table procat.retail.silver_order_lines as
select
    invoice,
    stock_code,
    description,
    try_cast(quantity as int) as quantity,
    try_cast(invoice_date as timestamp) as invoice_date,
    try_cast(price as decimal(12,3)) as price,
    cast(try_cast(customer_id as double) as int) as customer_id,
    country,
    try_cast(quantity as int) * try_cast(price as decimal(12,3)) as line_amount
from checked_rows
where row_status = 'valid';

## Step 3: save the rejected rows with the reason

In [ ]:
%sql
create or replace table procat.retail.silver_rejected_lines as
select
    invoice,
    stock_code,
    description,
    quantity,
    invoice_date,
    price,
    customer_id,
    country,
    row_status as reject_reason
from checked_rows
where row_status <> 'valid';

## Check 1: how many rows were rejected, and why

In [ ]:
%sql
select
    reject_reason,
    count(*) as row_count
from procat.retail.silver_rejected_lines
group by reject_reason
order by row_count desc;

## Check 2: do the row counts add up?

Bronze rows should equal Silver rows plus rejected rows. The last column should say true.

In [ ]:
%sql
select
    b.bronze_rows,
    s.silver_rows,
    r.rejected_rows,
    b.bronze_rows = s.silver_rows + r.rejected_rows as counts_match
from (select count(*) as bronze_rows from procat.retail.bronze_order_lines) as b
cross join (select count(*) as silver_rows from procat.retail.silver_order_lines) as s
cross join (select count(*) as rejected_rows from procat.retail.silver_rejected_lines) as r;

## Check 3: quality of the Silver table

No value should have failed to convert to a number or a date. Rows with a price of zero are kept (free items add no revenue) but counted here so they are visible.

In [ ]:
%sql
select
    count(*) as silver_rows,
    count(distinct customer_id) as customers,
    count(distinct invoice) as orders,
    round(sum(line_amount), 2) as total_revenue,
    sum(case when quantity is null then 1 else 0 end) as bad_quantity,
    sum(case when price is null then 1 else 0 end) as bad_price,
    sum(case when invoice_date is null then 1 else 0 end) as bad_date,
    sum(case when price = 0 then 1 else 0 end) as zero_price_rows,
    min(invoice_date) as first_order,
    max(invoice_date) as last_order
from procat.retail.silver_order_lines;